# POE_DETAIL - Bronze Layer

This notebook ingests the MIMIC-IV `poe_detail` source table and stores it as a Bronze Delta table.

A small source check is completed before ingestion to confirm its structure and relationship back to the main `POE` table.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"

SOURCE_PATH = "s3://mimic-iii-demo-bucket/mimic-iv-dataset-demo-2.2-folder/hosp/poe_detail.csv"

poe_detail = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

print(f"Rows: {poe_detail.count():,}")
print(f"Columns: {len(poe_detail.columns)}")

poe_detail.printSchema()

### Source Check

`POE_DETAIL` contains additional field-level information for records in the main `POE` table.

The key identifiers are checked for missing values, and `poe_id` is validated against the Bronze `POE` table to confirm the relationship between the two datasets.

In [0]:
# Check key fields
for column in ["poe_id", "poe_seq", "subject_id", "field_name"]:
    nulls = poe_detail.filter(F.col(column).isNull()).count()
    print(f"{column}: {nulls:,} nulls")

# Check POE relationship
poe_bronze = spark.table("workspace.bronze.poe_raw")

unmatched = (
    poe_detail
    .select("poe_id")
    .distinct()
    .join(
        poe_bronze.select("poe_id").distinct(),
        on="poe_id",
        how="left_anti"
    )
    .count()
)

print(f"\nUnmatched poe_id values: {unmatched:,}")

### Source Check

The key identifiers in `POE_DETAIL` are complete, and all `poe_id` values successfully match records in the main `POE` table.

This confirms the relationship between the two datasets is intact, so the source table is ready for Bronze ingestion.

In [0]:
# Add ingestion metadata
poe_detail_bronze = (
    poe_detail
    .withColumn("_source_file", F.lit("poe_detail.csv"))
    .withColumn("_source_module", F.lit("hosp"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as Delta table
(
    poe_detail_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.poe_detail_raw")
)

print("Created: workspace.bronze.poe_detail_raw")

In [0]:
source_count = poe_detail.count()
bronze_count = spark.table("workspace.bronze.poe_detail_raw").count()

print(f"Source: {source_count:,}")
print(f"Bronze: {bronze_count:,}")
print(f"Match: {source_count == bronze_count}")

### Bronze Result

`POE_DETAIL` was successfully written to the Bronze layer with all source records preserved.

The key identifiers are complete, all `poe_id` values match the main `POE` table, and the Bronze row count matches the source.
